# 第三章：机械手臂顺运动学 — 高级交互式仿真

> 台大林沛群教授《机器人学》第三章

| Part | 内容 |
|------|------|
| 1 | DH 四步分解动画 |
| 2 | RP 极坐标机械臂 |
| 3 | RRR 3D（含坐标系）+ MuJoCo 渲染 |
| 4 | PUMA 560 六自由度顺运动学 |
| 5 | Kinematic Mapping：关节空间 → 笛卡尔空间 |
| 6 | 工作空间 3D 可视化 |
| 7 | RRRP vs PRRR 构型对比 |

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import mujoco
import warnings
from pathlib import Path
from ipywidgets import FloatSlider, Dropdown, interactive_output, VBox, HBox
from IPython.display import display
from matplotlib import font_manager as fm

warnings.filterwarnings('ignore')
%matplotlib inline

# CJK Font
def _pick_font():
    want = ['Microsoft YaHei', 'Microsoft YaHei UI', 'SimHei', 'NSimSun',
            'Microsoft JhengHei', 'DejaVu Sans']
    installed = {f.name for f in fm.fontManager.ttflist}
    for n in want:
        if n in installed: 
            return n
    return 'DejaVu Sans'

CJK = _pick_font()
plt.rcParams.update({'font.family': CJK, 'font.sans-serif': [CJK, 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'figure.dpi': 110})

# Modified DH 变换矩阵
# 顺序 (a, alpha_deg, d, theta_deg) 与林教授 DH 表一致
# ^{i-1}T_i = Rx(α) · Tx(a) · Rz(θ) · Tz(d)
def mdh(a, alpha_deg, d, theta_deg):
    α = np.radians(alpha_deg)
    θ = np.radians(theta_deg)
    ca, sa = np.cos(α), np.sin(α)
    ct, st = np.cos(θ), np.sin(θ)
    return np.array([
        [ ct,    -st,      0,    a    ],
        [ st*ca,  ct*ca,  -sa, -d*sa ],
        [ st*sa,  ct*sa,   ca,  d*ca ],
        [ 0,      0,       0,   1    ]
    ])

def fk(dh_rows):
    """连乘 Modified DH 链，返回各坐标系 T 列表（含基座 I）"""
    T, frames = np.eye(4), [np.eye(4)]
    for row in dh_rows:
        T = T @ mdh(*row)
        frames.append(T.copy())
    return frames

# 3D 绘图工具
RC = ['#e74c3c', '#27ae60', '#2980b9']  # R G B

def draw_frame(ax, T, scale=0.12, labels=('x','y','z'), alpha=1.0, lw=2):
    o, R = T[:3, 3], T[:3, :3]
    for i, (c, lbl) in enumerate(zip(RC, labels)):
        v = R[:, i] * scale
        ax.quiver(*o, *v, color=c, alpha=alpha, arrow_length_ratio=0.22, linewidth=lw)
        if lbl:
            ax.text(*(o + R[:, i]*scale*1.4), lbl, color=c, fontsize=8, fontweight='bold')

def draw_arm(ax, frames, lc='#4a90d9', jc='#c0392b', lw=4):
    pts = np.array([f[:3, 3] for f in frames])
    for i in range(len(pts)-1):
        ax.plot(*zip(pts[i], pts[i+1]), color=lc, lw=lw, solid_capstyle='round')
    ax.scatter(*pts[:-1].T, color=jc, s=55, zorder=5, depthshade=False)
    ax.scatter(*pts[-1],    color='#e74c3c', s=160, zorder=6, marker='*', depthshade=False)
    return pts

def setup_3d(ax, lim=0.8, title='', elev=20, azim=40):
    ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_zlim(-lim,lim)
    ax.set_xlabel('X',fontsize=9); ax.set_ylabel('Y',fontsize=9); ax.set_zlabel('Z',fontsize=9)
    ax.set_box_aspect([1,1,1]); ax.view_init(elev=elev, azim=azim)
    if title: ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.xaxis.pane.fill = ax.yaxis.pane.fill = ax.zaxis.pane.fill = False

# MuJoCo 工具
def mj_render(xml, qpos_rad, cam='main', w=700, h=480):
    """给定 MJCF XML 和关节角（弧度），返回 RGB 图像 ndarray"""
    m = mujoco.MjModel.from_xml_string(xml)
    d = mujoco.MjData(m)
    n = min(len(qpos_rad), m.nq)
    d.qpos[:n] = np.asarray(qpos_rad)[:n]
    mujoco.mj_forward(m, d)
    r = mujoco.Renderer(m, height=h, width=w)
    r.update_scene(d, camera=cam)
    return r.render()

def show_pixels(pixels, title='', figsize=(8,5)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(pixels); ax.axis('off')
    if title: ax.set_title(title, fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

print(f'✓ 加载完成  |  字体: {CJK}  |  MuJoCo {mujoco.__version__}')

✓ 加载完成  |  字体: Microsoft YaHei  |  MuJoCo 3.6.0


---
## Part 1：DH 四步分解动画

一次 Modified DH 变换分四步完成：

$${}^{i-1}_iT = \underbrace{R_x(\alpha_{i-1})}_{\text{Step 1}} \cdot \underbrace{T_x(a_{i-1})}_{\text{Step 2}} \cdot \underbrace{R_z(\theta_i)}_{\text{Step 3}} \cdot \underbrace{T_z(d_i)}_{\text{Step 4}}$$

拖动滑块，观察每一步如何改变坐标系。

In [4]:
_s_a     = FloatSlider(value=0.6, min=0.0, max=1.2, step=0.1,  description='a')
_s_alpha = FloatSlider(value=30,  min=-90,  max=90,  step=15,   description='α (deg)')
_s_theta = FloatSlider(value=45,  min=-180, max=180, step=15,   description='θ (deg)')
_s_d     = FloatSlider(value=0.3, min=-0.8, max=0.8, step=0.1,  description='d')

def show_dh_steps(a, alpha, theta, d):
    plt.close('all')
    # 四步中间状态
    T0 = np.eye(4)
    T1 = T0 @ mdh(0, alpha, 0, 0)    # step1: Rx(α)
    T2 = T1 @ mdh(a, 0,     0, 0)    # step2: Tx(a)
    T3 = T2 @ mdh(0, 0,     0, theta)# step3: Rz(θ)
    T4 = T3 @ mdh(0, 0,     d, 0)    # step4: Tz(d)

    steps = [(T0, 'Step0: 基座 {i-1}'),
             (T1, f'Step1: Rx(α={alpha:.0f}°)'),
             (T2, f'Step2: Tx(a={a:.1f})'),
             (T3, f'Step3: Rz(θ={theta:.0f}°)'),
             (T4, f'Step4: Tz(d={d:.1f})  →  坐标系 {{i}}')]

    fig = plt.figure(figsize=(16, 4))
    for k, (T, title) in enumerate(steps):
        ax = fig.add_subplot(1, 5, k+1, projection='3d')
        # 始终画出基座（灰色，浅）
        draw_frame(ax, np.eye(4), scale=0.4, alpha=0.2,
                   labels=(None,None,None), lw=1)
        draw_frame(ax, T, scale=0.55, labels=('x','y','z'), lw=2)
        # 在 step>=2 时，画一条从前一步到当前步的平移线
        if k >= 2:
            prev_T = steps[k-1][0]
            ax.plot(*zip(prev_T[:3,3], T[:3,3]), '--', color='gray', alpha=0.6, lw=1.5)
        setup_3d(ax, lim=1.0, title=title, elev=18, azim=35)

    plt.suptitle('Modified DH 四步分解  (每步坐标系用 RGB = xyz 表示)',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

_out1 = interactive_output(show_dh_steps,
    {'a': _s_a, 'alpha': _s_alpha, 'theta': _s_theta, 'd': _s_d})
display(VBox([HBox([_s_a, _s_alpha]), HBox([_s_theta, _s_d]), _out1]))

---
## Part 2：RP 极坐标机械臂

- **Joint 1（R）**：转动关节，关节变量 $\theta$
- **Joint 2（P）**：移动关节，关节变量 $r$（沿连杆方向伸缩）

$$x = r\cos\theta, \quad y = r\sin\theta$$

关节空间是个矩形 $(\theta, r)$，笛卡尔空间里是**扇环**。

In [ ]:
_rp_theta = FloatSlider(value=45, min=-180, max=180, step=5, description='θ (deg)')
_rp_r     = FloatSlider(value=0.8, min=0.2, max=1.5, step=0.05, description='r')

def show_rp(theta, r):
    plt.close('all')
    t = np.radians(theta)
    ee = np.array([r * np.cos(t), r * np.sin(t)])

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5))

    # 机械臂结构图
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-1.8, 1.8)
    ax1.set_ylim(-1.8, 1.8)
    ax1.set_title('RP 机械臂当前姿态', fontsize=11, fontweight='bold')
    # 画出当前臂
    ax1.annotate('', xy=ee, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color='#2980b9', lw=3))
    ax1.scatter(*ee, color='crimson', s=120, zorder=5)
    ax1.scatter(0, 0, color='#333', s=80, zorder=5)
    ax1.text(ee[0]+0.05, ee[1]+0.05, f'EE=({ee[0]:.2f},{ee[1]:.2f})', fontsize=9)
    # 画角度弧
    arc_t = np.linspace(0, t, 60)
    arc_r = 0.25
    ax1.plot(arc_r*np.cos(arc_t), arc_r*np.sin(arc_t), 'k-', lw=1.5)
    ax1.text(0.28*np.cos(t/2), 0.28*np.sin(t/2), f'θ={theta:.0f}°', fontsize=9)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')

    # θ 变化轨迹（r 固定）
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-1.8, 1.8)
    ax2.set_ylim(-1.8, 1.8)
    ax2.set_title('θ 变化（r 固定）→ 圆弧', fontsize=11)
    ts = np.linspace(-np.pi, np.pi, 200)
    ax2.plot(r*np.cos(ts), r*np.sin(ts), color='#8e44ad', lw=2, alpha=0.7, label=f'r={r:.2f}')
    ax2.scatter(*ee, color='crimson', s=80, zorder=5)
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.legend()

    # r 变化轨迹（θ 固定）
    ax3.set_aspect('equal'); ax3.grid(True, alpha=0.3)
    ax3.set_xlim(-1.8, 1.8); ax3.set_ylim(-1.8, 1.8)
    ax3.set_title('r 变化（θ 固定）→ 射线', fontsize=11)
    rs = np.linspace(0.2, 1.5, 100)
    ax3.plot(rs*np.cos(t), rs*np.sin(t), color='#e67e22', lw=2, alpha=0.8, label=f'θ={theta:.0f}°')
    ax3.scatter(*ee, color='crimson', s=80, zorder=5)
    ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.legend()

    plt.suptitle(f'RP 极坐标机械臂  |  θ={theta:.0f}°   r={r:.2f}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out2 = interactive_output(show_rp, {'theta': _rp_theta, 'r': _rp_r})
display(VBox([HBox([_rp_theta, _rp_r]), _out2]))

---
## Part 3：RRR 机械臂（3D 坐标系可视化 + MuJoCo 渲染）

三关节全部为转动关节，所有轴平行（$\alpha=0$，$d=0$）。

**DH 参数表**：

| $i$ | $\alpha_{i-1}$ | $a_{i-1}$ | $d_i$ | $\theta_i$ |
|---|---|---|---|---|
| 1 | 0° | 0 | 0 | $\theta_1$ |
| 2 | 0° | $L_1$ | 0 | $\theta_2$ |
| 3 | 0° | $L_2$ | 0 | $\theta_3$ |
| EE | 0° | $L_3$ | 0 | 0 |

In [ ]:
L1, L2, L3 = 0.50, 0.40, 0.30

_rrr_t1 = FloatSlider(value=30,  min=-180, max=180, step=5, description='θ₁ (°)')
_rrr_t2 = FloatSlider(value=60,  min=-180, max=180, step=5, description='θ₂ (°)')
_rrr_t3 = FloatSlider(value=-45, min=-180, max=180, step=5, description='θ₃ (°)')

def rrr_fk(t1, t2, t3):
    dh = [(0, 0, 0, t1), (L1, 0, 0, t2), (L2, 0, 0, t3), (L3, 0, 0, 0)]
    return fk(dh)

def show_rrr_mpl(t1, t2, t3):
    plt.close('all')
    frames = rrr_fk(t1, t2, t3)
    ee = frames[-1][:3, 3]
    phi = t1 + t2 + t3

    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

    # ── 左：3D 透视（含所有坐标系）──
    ax1 = fig.add_subplot(gs[0], projection='3d')
    draw_arm(ax1, frames, lw=5)
    frame_labels = [('x₀','y₀','z₀'), ('x₁','y₁','z₁'), ('x₂','y₂','z₂'),
                    ('x₃','y₃','z₃'), ('xe','ye','ze')]
    for i, T in enumerate(frames):
        alp = 0.35 if i == 0 else (0.5 if i < 3 else 1.0)
        draw_frame(ax1, T, scale=0.11, labels=frame_labels[i], alpha=alp)
    lim = L1+L2+L3+0.05
    setup_3d(ax1, lim=lim, title='坐标系链（透视）', elev=28, azim=42)

    # ── 中：俯视 XY ──
    ax2 = fig.add_subplot(gs[1], projection='3d')
    draw_arm(ax2, frames, lw=5)
    draw_frame(ax2, frames[-1], scale=0.13, labels=('xe','ye','ze'))
    setup_3d(ax2, lim=lim, title='俯视（XY 平面）', elev=88, azim=90)

    # ── 右：信息面板 ──
    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    txt = (f"DH 参数（α=0, d=0）\n"
           f"  i   a      θ\n"
           f"  1   0      {t1:.0f}°\n"
           f"  2   {L1:.2f}   {t2:.0f}°\n"
           f"  3   {L2:.2f}   {t3:.0f}°\n"
           f"  EE  {L3:.2f}   0°\n\n"
           f"末端坐标：\n"
           f"  x = {ee[0]:.4f}\n"
           f"  y = {ee[1]:.4f}\n"
           f"  z = {ee[2]:.4f}\n\n"
           f"末端姿态角：\n"
           f"  φ = θ₁+θ₂+θ₃ = {phi:.0f}°\n\n"
           f"几何法验证：\n"
           f"  x = {L1*np.cos(np.radians(t1))+L2*np.cos(np.radians(t1+t2))+L3*np.cos(np.radians(phi)):.4f}\n"
           f"  y = {L1*np.sin(np.radians(t1))+L2*np.sin(np.radians(t1+t2))+L3*np.sin(np.radians(phi)):.4f}")
    ax3.text(0.05, 0.95, txt, transform=ax3.transAxes, va='top',
             fontsize=10, fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#eef3fa', alpha=0.92))

    plt.suptitle(f'RRR 平面机械臂  |  θ₁={t1:.0f}°  θ₂={t2:.0f}°  θ₃={t3:.0f}°',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

_out3 = interactive_output(show_rrr_mpl, {'t1': _rrr_t1, 't2': _rrr_t2, 't3': _rrr_t3})
display(VBox([HBox([_rrr_t1, _rrr_t2, _rrr_t3]), _out3]))

In [ ]:
# RRR MuJoCo 渲染
def _rrr_xml(l1=0.50, l2=0.40, l3=0.30, scale=2.0):
    """scale: 整体放大倍数，默认 2× 让机器人在画面中更突出"""
    l1, l2, l3 = l1*scale, l2*scale, l3*scale
    r1, r2, r3 = 0.060*scale, 0.050*scale, 0.038*scale
    base_h = 0.12 * scale
    scene  = (l1+l2+l3) * 1.6  # 地板半尺寸
    cam_d  = (l1+l2+l3) * 1.4  # 相机距离

    return f"""<mujoco model="rrr">
  <compiler angle="radian"/>
  <option gravity="0 0 0"/>
  <visual>
    <quality shadowsize="4096"/>
    <map shadowscale="0.5" znear="0.01"/>
  </visual>
  <asset>
    <texture name="chk" type="2d" builtin="checker"
             rgb1=".14 .18 .24" rgb2=".22 .27 .34" width="1024" height="1024"/>
    <material name="floor_mat" texture="chk" texrepeat="6 6"
              reflectance=".20" shininess="0.1"/>
    <material name="base_mat"  rgba=".25 .27 .30 1" shininess="0.6" reflectance=".3"/>
    <material name="link1_mat" rgba=".16 .50 .82 1" shininess="0.8" reflectance=".4"/>
    <material name="link2_mat" rgba=".14 .70 .42 1" shininess="0.8" reflectance=".4"/>
    <material name="link3_mat" rgba=".90 .58 .10 1" shininess="0.7" reflectance=".3"/>
    <material name="joint_mat" rgba=".80 .15 .15 1" shininess="0.9" reflectance=".5"/>
    <material name="ee_mat"    rgba=".95 .10 .10 1" shininess="1.0" reflectance=".6"/>
  </asset>
  <worldbody>
    <light name="sun"  pos="3 -2 6"   dir="-.4  .3 -1"  diffuse=".90 .90 .95" specular=".4 .4 .4" castshadow="true"/>
    <light name="fill" pos="-2  3 3"  dir=".5  -.6 -.5" diffuse=".30 .30 .40" specular="0  0  0 " castshadow="false"/>
    <light name="back" pos="0  -3 2"  dir="0    .8 -.5" diffuse=".20 .20 .25" specular="0  0  0 " castshadow="false"/>
    <geom name="floor" type="plane" size="{scene:.2f} {scene:.2f} .02" material="floor_mat"/>
    <!-- 底座 -->
    <geom type="cylinder" size="{r1*1.8:.4f} {base_h*0.4:.4f}"
          pos="0 0 {base_h*0.4:.4f}" material="base_mat"/>
    <geom type="cylinder" size="{r1*0.7:.4f} {base_h*0.5:.4f}"
          pos="0 0 {base_h*0.8:.4f}" material="base_mat"/>
    <!-- 连杆 1 -->
    <body name="L1" pos="0 0 {base_h:.4f}">
      <joint name="j1" type="hinge" axis="0 0 1"/>
      <geom type="sphere"  size="{r1*1.4:.4f}"                      material="joint_mat"/>
      <geom type="capsule" fromto="0 0 0 {l1:.4f} 0 0" size="{r1:.4f}" material="link1_mat"/>
      <!-- 连杆 2 -->
      <body name="L2" pos="{l1:.4f} 0 0">
        <joint name="j2" type="hinge" axis="0 0 1"/>
        <geom type="sphere"  size="{r2*1.4:.4f}"                      material="joint_mat"/>
        <geom type="capsule" fromto="0 0 0 {l2:.4f} 0 0" size="{r2:.4f}" material="link2_mat"/>
        <!-- 连杆 3 -->
        <body name="L3" pos="{l2:.4f} 0 0">
          <joint name="j3" type="hinge" axis="0 0 1"/>
          <geom type="sphere"  size="{r3*1.4:.4f}"                      material="joint_mat"/>
          <geom type="capsule" fromto="0 0 0 {l3:.4f} 0 0" size="{r3:.4f}" material="link3_mat"/>
          <!-- 末端执行器 -->
          <geom type="sphere" size="{r3*2.2:.4f}" pos="{l3:.4f} 0 0" material="ee_mat"/>
        </body>
      </body>
    </body>
    <!-- 相机必须在 worldbody 内 (MuJoCo 3.x) -->
    <camera name="main" pos="{cam_d*0.90:.3f} {-cam_d*0.75:.3f} {cam_d*0.65:.3f}"
            xyaxes=".64 .77 0 -.37 .31 .88"/>
    <camera name="top"  pos="0 0 {cam_d*1.5:.3f}" xyaxes="1 0 0 0 1 0"/>
    <camera name="side" pos="{cam_d*1.2:.3f} 0 {cam_d*0.4:.3f}"
            xyaxes="0 1 0 -.32 0 .95"/>
  </worldbody>
</mujoco>"""

_RRR_XML = _rrr_xml(L1, L2, L3, scale=2.2)

_mj_t1 = FloatSlider(value=30,  min=-180, max=180, step=5, description='θ₁ (°)')
_mj_t2 = FloatSlider(value=60,  min=-180, max=180, step=5, description='θ₂ (°)')
_mj_t3 = FloatSlider(value=-45, min=-180, max=180, step=5, description='θ₃ (°)')
_mj_cam = __import__('ipywidgets').Dropdown(
    options=[('透视 (main)','main'), ('俯视 (top)','top'), ('侧视 (side)','side')],
    description='视角')

def show_rrr_mujoco(t1, t2, t3, cam):
    plt.close('all')
    q = np.radians([t1, t2, t3])
    px = mj_render(_RRR_XML, q, cam=cam, w=800, h=560)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(px); ax.axis('off')
    ax.set_title(f'RRR MuJoCo 渲染  |  θ₁={t1:.0f}°  θ₂={t2:.0f}°  θ₃={t3:.0f}°',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out3b = interactive_output(show_rrr_mujoco,
    {'t1': _mj_t1, 't2': _mj_t2, 't3': _mj_t3, 'cam': _mj_cam})
display(VBox([HBox([_mj_t1, _mj_t2, _mj_t3, _mj_cam]), _out3b]))

---
## Part 4：PUMA 560 六自由度顺运动学

PUMA 560 是课程中标准的 6-DOF 空间机械臂例题（Craig 教材 Table 3.4）。

**Modified DH 参数表**：

| $i$ | $\alpha_{i-1}$ | $a_{i-1}$ (m) | $d_i$ (m) | $\theta_i$ |
|---|---|---|---|---|
| 1 | 0° | 0 | 0 | $\theta_1$ |
| 2 | −90° | 0 | 0 | $\theta_2$ |
| 3 | 0° | 0.4318 | 0 | $\theta_3$ |
| 4 | −90° | −0.0203 | 0.4331 | $\theta_4$ |
| 5 | 90° | 0 | 0 | $\theta_5$ |
| 6 | −90° | 0 | 0 | $\theta_6$ |

关节 1–3 决定**腕部位置**，关节 4–6 决定**腕部姿态**（三轴交于腕点）。

In [8]:
PUMA_DH_BASE = [
    (0,      0,   0,      0),  # i=1  a, α, d, θ（θ为变量位）
    (0,    -90,   0,      0),  # i=2
    (0.4318, 0,   0,      0),  # i=3
    (-0.0203,-90, 0.4331, 0),  # i=4
    (0,     90,   0,      0),  # i=5
    (0,    -90,   0,      0),  # i=6
]

_p1 = FloatSlider(value=0,   min=-180, max=180, step=5, description='θ₁ (°)')
_p2 = FloatSlider(value=90,  min=-135, max=135, step=5, description='θ₂ (°)')
_p3 = FloatSlider(value=-90, min=-135, max=135, step=5, description='θ₃ (°)')
_p4 = FloatSlider(value=0,   min=-180, max=180, step=5, description='θ₄ (°)')
_p5 = FloatSlider(value=90,  min=-135, max=135, step=5, description='θ₅ (°)')
_p6 = FloatSlider(value=0,   min=-180, max=180, step=5, description='θ₆ (°)')

def puma_fk(t1, t2, t3, t4, t5, t6):
    thetas = [t1, t2, t3, t4, t5, t6]
    rows = [(PUMA_DH_BASE[i][0], PUMA_DH_BASE[i][1],
             PUMA_DH_BASE[i][2], thetas[i]) for i in range(6)]
    return fk(rows)

def show_puma(t1, t2, t3, t4, t5, t6):
    plt.close('all')
    frames = puma_fk(t1, t2, t3, t4, t5, t6)
    T06 = frames[-1]
    ee_pos = T06[:3, 3]
    ee_R   = T06[:3, :3]

    fig = plt.figure(figsize=(15, 6))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05, width_ratios=[1.2, 1.2, 0.9])

    # ── 左：3D 透视 ──
    ax1 = fig.add_subplot(gs[0], projection='3d')
    # 颜色从基座到腕部渐变
    cm = plt.cm.plasma(np.linspace(0.15, 0.90, len(frames)))
    draw_arm(ax1, frames, lc='#555', lw=5)
    for i, T in enumerate(frames):
        scale = 0.07 if i < 4 else 0.09
        draw_frame(ax1, T, scale=scale, alpha=0.55 if i < 4 else 1.0,
                   labels=(None,None,None) if i < 4 else ('xe','ye','ze'))
    # 腕点（frame 4）特别标出
    w_pos = frames[4][:3, 3]
    ax1.scatter(*w_pos, color='gold', s=80, zorder=7, depthshade=False)
    ax1.text(*w_pos, '  腕点', fontsize=8, color='goldenrod')
    setup_3d(ax1, lim=0.7, title='PUMA 560 透视图', elev=22, azim=48)

    # ── 中：侧视 XZ ──
    ax2 = fig.add_subplot(gs[1], projection='3d')
    draw_arm(ax2, frames, lw=5)
    draw_frame(ax2, frames[-1], scale=0.09, labels=('xe','ye','ze'))
    setup_3d(ax2, lim=0.7, title='侧视（XZ 平面）', elev=0, azim=0)

    # ── 右：数值面板 ──
    ax3 = fig.add_subplot(gs[2]); ax3.axis('off')
    # 计算 Euler angles ZYX from R
    R = ee_R
    beta  = np.degrees(np.arctan2(-R[2,0], np.sqrt(R[0,0]**2+R[1,0]**2)))
    alpha = np.degrees(np.arctan2(R[1,0]/np.cos(np.radians(beta)),
                                  R[0,0]/np.cos(np.radians(beta))))
    gamma = np.degrees(np.arctan2(R[2,1]/np.cos(np.radians(beta)),
                                  R[2,2]/np.cos(np.radians(beta))))
    txt = (f"^0T_6（末端位姿）\n\n"
           f"位置 p：\n"
           f"  x = {ee_pos[0]:.4f} m\n"
           f"  y = {ee_pos[1]:.4f} m\n"
           f"  z = {ee_pos[2]:.4f} m\n\n"
           f"姿态 R（ZYX Euler）：\n"
           f"  α(yaw)  = {alpha:.1f}°\n"
           f"  β(pitch)= {beta:.1f}°\n"
           f"  γ(roll) = {gamma:.1f}°\n\n"
           f"腕点 p_w = frames[4]：\n"
           f"  {np.round(frames[4][:3,3],4)}")
    ax3.text(0.04, 0.96, txt, transform=ax3.transAxes, va='top',
             fontsize=9.5, fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#fdf6e3', alpha=0.95))

    plt.suptitle(f'PUMA 560  |  θ=[ {t1:.0f}°, {t2:.0f}°, {t3:.0f}°, {t4:.0f}°, {t5:.0f}°, {t6:.0f}° ]',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out4 = interactive_output(show_puma,
    {'t1':_p1,'t2':_p2,'t3':_p3,'t4':_p4,'t5':_p5,'t6':_p6})
display(VBox([HBox([_p1,_p2,_p3]), HBox([_p4,_p5,_p6]), _out4]))

---
## Part 5：Kinematic Mapping（关节空间 → 笛卡尔空间）

林教授强调：**关节空间里的直线，映射到笛卡尔空间不一定是直线。**

下面同时展示四条轨迹：线性插值 / 圆弧 / 摆动 / 螺旋形，观察两个空间的对应关系。

In [9]:
def fk_planar_ee(t1_deg, t2_deg, l1=1.2, l2=0.9):
    t1, t2 = np.radians(t1_deg), np.radians(t2_deg)
    x = l1*np.cos(t1) + l2*np.cos(t1+t2)
    y = l1*np.sin(t1) + l2*np.sin(t1+t2)
    return x, y

_km_t1s = FloatSlider(value=10,  min=-160, max=160, step=5, description='θ₁ start')
_km_t2s = FloatSlider(value=20,  min=-160, max=160, step=5, description='θ₂ start')
_km_t1e = FloatSlider(value=80,  min=-160, max=160, step=5, description='θ₁ end')
_km_t2e = FloatSlider(value=-40, min=-160, max=160, step=5, description='θ₂ end')

def show_kinematic_map(t1s, t2s, t1e, t2e):
    plt.close('all')
    N = 120
    ts = np.linspace(0, 1, N)

    # 四种关节空间轨迹
    traj_js = {
        '线性插值': (t1s + (t1e-t1s)*ts,
                    t2s + (t2e-t2s)*ts),
        '圆弧（关节空间）': (t1s + (t1e-t1s)*ts,
                            t2s + (t2e-t2s)*np.sin(ts*np.pi/2)),
        '摆动': (t1s + (t1e-t1s)*ts,
                  t2s + (t2e-t2s)*ts + 30*np.sin(ts*4*np.pi)),
        '慢起快落': (t1s + (t1e-t1s)*ts**2,
                     t2s + (t2e-t2s)*ts**2),
    }
    colors = ['#2980b9','#e74c3c','#27ae60','#8e44ad']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # 关节空间
    ax1.set_title('关节空间 (θ₁, θ₂)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('θ₁ (°)'); ax1.set_ylabel('θ₂ (°)')
    ax1.grid(True, alpha=0.3)
    for (name, (q1, q2)), c in zip(traj_js.items(), colors):
        ax1.plot(q1, q2, color=c, lw=2.5, label=name)
        ax1.scatter([q1[0],q1[-1]], [q2[0],q2[-1]],
                    color=['#2ecc71','#e74c3c'], s=60, zorder=5)
    ax1.legend(fontsize=9, loc='best')

    # 笛卡尔空间
    ax2.set_title('笛卡尔空间（末端轨迹）', fontsize=12, fontweight='bold')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y')
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    # 画工作空间边界（参考）
    l1, l2 = 1.2, 0.9
    for R, sty in [(l1+l2,'--'), (abs(l1-l2),':')]:
        tc = np.linspace(0, 2*np.pi, 200)
        ax2.plot(R*np.cos(tc), R*np.sin(tc), sty, color='gray', alpha=0.4, lw=1)
    for (name, (q1, q2)), c in zip(traj_js.items(), colors):
        xs, ys = zip(*[fk_planar_ee(a, b, l1, l2) for a,b in zip(q1,q2)])
        ax2.plot(xs, ys, color=c, lw=2.5, label=name)
        ax2.scatter([xs[0],xs[-1]], [ys[0],ys[-1]],
                    color=['#2ecc71','#e74c3c'], s=60, zorder=5)
    ax2.legend(fontsize=9, loc='best')

    plt.suptitle('Kinematic Mapping：相同起终点，四种关节空间路径 → 四条不同末端轨迹',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out5 = interactive_output(show_kinematic_map,
    {'t1s':_km_t1s,'t2s':_km_t2s,'t1e':_km_t1e,'t2e':_km_t2e})
display(VBox([HBox([_km_t1s,_km_t2s]), HBox([_km_t1e,_km_t2e]), _out5]))

---
## Part 6：工作空间 3D 可视化

工作空间 = 末端执行器**所有可达位置**的集合。

改变各关节范围，观察工作空间形状如何变化。

In [ ]:
_ws_q1r = FloatSlider(value=120, min=10, max=180, step=10, description='±q1 范围')
_ws_q2r = FloatSlider(value=150, min=10, max=180, step=10, description='±q2 范围')
_ws_q3r = FloatSlider(value=150, min=10, max=180, step=10, description='±q3 范围')
_ws_n   = FloatSlider(value=40,  min=15, max=70,  step=5,  description='采样密度')

def show_workspace(q1r, q2r, q3r, n):
    plt.close('all')
    n = int(n)
    l1, l2, l3 = 0.5, 0.4, 0.3
    q1s = np.linspace(-q1r, q1r, n)
    q2s = np.linspace(-q2r, q2r, n)
    q3s = np.linspace(-q3r, q3r, n)

    # 均匀采样（降低计算量，用步长跳采）
    pts, phis = [], []
    for q1 in q1s[::2]:
        for q2 in q2s[::2]:
            for q3 in q3s[::2]:
                rows = [(0,0,0,q1),(l1,0,0,q2),(l2,0,0,q3),(l3,0,0,0)]
                ee = fk(rows)[-1][:3, 3]
                pts.append(ee)
                phis.append((q1+q2+q3) % 360)
    pts = np.array(pts)
    phis = np.array(phis)

    fig = plt.figure(figsize=(14, 6))

    # 3D 工作空间（按末端姿态角着色）
    ax1 = fig.add_subplot(121, projection='3d')
    sc = ax1.scatter(pts[:,0], pts[:,1], pts[:,2],
                     c=phis, cmap='hsv', s=3, alpha=0.25)
    plt.colorbar(sc, ax=ax1, label='末端姿态角 φ (°)', shrink=0.7)
    ax1.set_title('3D 工作空间（颜色=φ角）', fontsize=11, fontweight='bold')
    ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
    lim = l1+l2+l3+0.05
    ax1.set_xlim(-lim,lim); ax1.set_ylim(-lim,lim); ax1.set_zlim(-lim,lim)
    ax1.set_box_aspect([1,1,1])

    # XY 俯视截面
    ax2 = fig.add_subplot(122)
    ax2.scatter(pts[:,0], pts[:,1], c=phis, cmap='hsv', s=3, alpha=0.2)
    ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
    ax2.set_title('XY 平面工作空间截面', fontsize=11, fontweight='bold')
    ax2.set_xlabel('X'); ax2.set_ylabel('Y')
    for R, sty in [(l1+l2+l3,'--'), (abs(l1-l2-l3),':')]:
        tc = np.linspace(0, 2*np.pi, 300)
        ax2.plot(R*np.cos(tc), R*np.sin(tc), sty, color='black', alpha=0.5, lw=1.5)

    plt.suptitle(f'2R+R 工作空间  |  q1=±{q1r:.0f}°  q2=±{q2r:.0f}°  q3=±{q3r:.0f}°',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out6 = interactive_output(show_workspace,
    {'q1r':_ws_q1r,'q2r':_ws_q2r,'q3r':_ws_q3r,'n':_ws_n})
display(VBox([HBox([_ws_q1r, _ws_q2r]), HBox([_ws_q3r, _ws_n]), _out6]))

---
## Part 7：RRRP vs PRRR 构型对比

| 构型 | 特点 | 典型应用 |
|------|------|----------|
| **RRRP** | 末端伸缩（焊枪/喷涂） | 焊接机器人末端执行器 |
| **PRRR** | 基座直线滑轨 + 三转动 | 悬挂式工业机器人 |

In [ ]:
_rp_t1 = FloatSlider(value=40,  min=-180, max=180, step=5,  description='θ₁')
_rp_t2 = FloatSlider(value=60,  min=-180, max=180, step=5,  description='θ₂')
_rp_t3 = FloatSlider(value=-30, min=-180, max=180, step=5,  description='θ₃')
_rp_d4 = FloatSlider(value=0.2, min=0.0,  max=0.6, step=0.05, description='d₄ (RRRP)')
_pp_d1 = FloatSlider(value=0.3, min=0.0,  max=0.8, step=0.05, description='d₁ (PRRR)')

LA, LB, LC = 0.45, 0.35, 0.25  # 三段连杆长

def show_rrrp_prrr(t1, t2, t3, d4, d1):
    plt.close('all')

    # ── RRRP：三转+末端伸缩（沿末端轴 z₃ 延伸）──
    rrrp_dh = [(0,0,0,t1), (LA,0,0,t2), (LB,0,0,t3), (LC,0,d4,0)]
    rrrp_frames = fk(rrrp_dh)

    # ── PRRR：基座沿 z 平移 + 三转 ──
    prrr_dh = [(0,0,d1,0), (0,0,0,t1), (LA,0,0,t2), (LB,0,0,t3), (LC,0,0,0)]
    prrr_frames = fk(prrr_dh)

    fig = plt.figure(figsize=(15, 6))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

    # ── 左：RRRP ──
    ax1 = fig.add_subplot(gs[0], projection='3d')
    draw_arm(ax1, rrrp_frames, lc='#2980b9', lw=4)
    draw_frame(ax1, rrrp_frames[-1], scale=0.10, labels=('xe','ye','ze'))
    # 用虚线标出 d4 伸缩部分
    p3 = rrrp_frames[-2][:3, 3]
    ee = rrrp_frames[-1][:3, 3]
    ax1.plot(*zip(p3, ee), '--', color='orange', lw=3, label=f'd₄={d4:.2f}')
    lim = LA+LB+LC+0.3
    setup_3d(ax1, lim=lim, title=f'RRRP  |  末端伸缩 d₄={d4:.2f}m', elev=22, azim=45)
    ax1.legend(fontsize=9)

    # ── 中：PRRR ──
    ax2 = fig.add_subplot(gs[1], projection='3d')
    draw_arm(ax2, prrr_frames, lc='#e74c3c', lw=4)
    draw_frame(ax2, prrr_frames[-1], scale=0.10, labels=('xe','ye','ze'))
    # 用虚线标出 d1 滑轨部分
    p0 = prrr_frames[0][:3, 3]
    p1 = prrr_frames[1][:3, 3]
    ax2.plot(*zip(p0, p1), '--', color='orange', lw=3, label=f'd₁={d1:.2f}')
    setup_3d(ax2, lim=lim, title=f'PRRR  |  基座滑轨 d₁={d1:.2f}m', elev=22, azim=45)
    ax2.legend(fontsize=9)

    # ── 右：末端轨迹对比（固定 d，扫描 θ₁）──
    ax3 = fig.add_subplot(gs[2], projection='3d')
    ax3.set_title('末端轨迹对比\n（θ₁ 从 -120° 到 120°）', fontsize=10, fontweight='bold')
    ts = np.linspace(-120, 120, 100)
    # RRRP 轨迹
    pts_rrrp = [fk([(0,0,0,a),(LA,0,0,t2),(LB,0,0,t3),(LC,0,d4,0)])[-1][:3,3] for a in ts]
    pts_rrrp = np.array(pts_rrrp)
    ax3.plot(pts_rrrp[:,0], pts_rrrp[:,1], pts_rrrp[:,2],
             color='#2980b9', lw=2, label='RRRP')
    # PRRR 轨迹
    pts_prrr = [fk([(0,0,d1,0),(0,0,0,a),(LA,0,0,t2),(LB,0,0,t3),(LC,0,0,0)])[-1][:3,3] for a in ts]
    pts_prrr = np.array(pts_prrr)
    ax3.plot(pts_prrr[:,0], pts_prrr[:,1], pts_prrr[:,2],
             color='#e74c3c', lw=2, label='PRRR')
    ax3.set_xlim(-lim,lim); ax3.set_ylim(-lim,lim); ax3.set_zlim(-lim,lim)
    ax3.set_box_aspect([1,1,1]); ax3.legend(fontsize=9)
    ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')

    plt.suptitle(f'RRRP vs PRRR  |  θ=({t1:.0f}°,{t2:.0f}°,{t3:.0f}°)  d₄={d4:.2f}  d₁={d1:.2f}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

_out7 = interactive_output(show_rrrp_prrr,
    {'t1':_rp_t1,'t2':_rp_t2,'t3':_rp_t3,'d4':_rp_d4,'d1':_pp_d1})
display(VBox([HBox([_rp_t1,_rp_t2,_rp_t3]), HBox([_rp_d4,_pp_d1]), _out7]))

---
## Part 8：批量导出静态图

In [12]:
def _out_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd/'Robotics_NTU'):
        if (base/'Chapter03_visualization.ipynb').exists():
            d = base/'images'; d.mkdir(exist_ok=True); return d
    d = cwd/'images'; d.mkdir(exist_ok=True); return d

OUTPUT_DIR = _out_dir()
print(f'导出目录: {OUTPUT_DIR}')

def _save(fig, name):
    p = OUTPUT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    return p

def export_all():
    l1, l2, l3 = 0.5, 0.4, 0.3
    paths = []

    # 1. DH 四步
    fig = plt.figure(figsize=(16,4))
    steps = [np.eye(4)]
    for fn in [lambda: mdh(0,30,0,0), lambda: mdh(0.6,0,0,0),
                lambda: mdh(0,0,0,45), lambda: mdh(0,0,0.3,0)]:
        steps.append(steps[-1] @ fn())
    titles = ['Step0','Step1: Rx(30°)','Step2: Tx(0.6)','Step3: Rz(45°)','Step4: Tz(0.3)']
    for k,(T,t) in enumerate(zip(steps,titles)):
        ax=fig.add_subplot(1,5,k+1,projection='3d')
        draw_frame(ax,T,scale=0.55,labels=('x','y','z'),lw=2)
        setup_3d(ax,lim=1.0,title=t)
    paths.append(_save(fig,'ch03_dh_steps.png'))

    # 2. RRR MuJoCo
    xml = _rrr_xml(l1,l2,l3)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for ax,cam,lbl in zip(axes,['main','top'],['透视','俯视']):
        ax.imshow(mj_render(xml,np.radians([30,60,-45]),cam=cam))
        ax.axis('off'); ax.set_title(f'RRR · {lbl}',fontsize=11)
    paths.append(_save(fig,'ch03_rrr_mujoco.png'))

    # 3. PUMA 560
    frames = puma_fk(0,90,-90,0,90,0)
    fig=plt.figure(figsize=(7,6))
    ax=fig.add_subplot(111,projection='3d')
    draw_arm(ax,frames,lw=5)
    draw_frame(ax,frames[-1],scale=0.08,labels=('xe','ye','ze'))
    setup_3d(ax,lim=0.65,title='PUMA 560 · θ=[0,90,-90,0,90,0]°',elev=22,azim=48)
    paths.append(_save(fig,'ch03_puma560.png'))

    # 4. Kinematic Mapping
    N=120; ts=np.linspace(0,1,N)
    t1s,t2s,t1e,t2e=10,20,80,-40
    trajs={'线性': (t1s+(t1e-t1s)*ts, t2s+(t2e-t2s)*ts),
           '摆动': (t1s+(t1e-t1s)*ts, t2s+(t2e-t2s)*ts+30*np.sin(ts*4*np.pi))}
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5))
    for (nm,(q1,q2)),c in zip(trajs.items(),['#2980b9','#e74c3c']):
        ax1.plot(q1,q2,color=c,lw=2,label=nm)
        xs,ys=zip(*[fk_planar_ee(a,b) for a,b in zip(q1,q2)])
        ax2.plot(xs,ys,color=c,lw=2,label=nm)
    ax1.legend(); ax1.grid(True,alpha=0.3); ax1.set_title('关节空间')
    ax2.legend(); ax2.grid(True,alpha=0.3); ax2.set_aspect('equal'); ax2.set_title('笛卡尔空间')
    paths.append(_save(fig,'ch03_kinematic_map.png'))

    # 5. Workspace
    pts=[]
    for q1 in np.linspace(-120,120,30):
        for q2 in np.linspace(-150,150,30):
            for q3 in np.linspace(-150,150,30):
                pts.append(fk([(0,0,0,q1),(l1,0,0,q2),(l2,0,0,q3),(l3,0,0,0)])[-1][:3,3])
    pts=np.array(pts)
    fig=plt.figure(figsize=(7,6))
    ax=fig.add_subplot(111,projection='3d')
    ax.scatter(pts[:,0],pts[:,1],pts[:,2],c=pts[:,2],cmap='plasma',s=2,alpha=0.2)
    ax.set_box_aspect([1,1,1]); ax.set_title('3D 工作空间')
    paths.append(_save(fig,'ch03_workspace3d.png'))

    print('导出完成：')
    for p in paths: print(f'  {p}')
    return paths

# 取消注释运行：
# export_all()

导出目录: d:\EI-Beginner\Robotics_NTU\images


---
## Part 9：方案一 — Meshcat 3D 可视化（浏览器 WebGL）

`meshcat` 在本地启动一个 WebSocket 服务器，在浏览器里渲染 Three.js 场景。  
拖动 ipywidgets 滑块 → Python 实时推送变换到浏览器。

In [13]:
import meshcat
import meshcat.geometry as mc_g
import meshcat.transformations as mc_tf
from ipywidgets import FloatSlider, VBox, HBox
from IPython.display import display

# ── 创建 Visualizer（自动开启本地服务器）──────────────────────────────────────
vis = meshcat.Visualizer()
print("浏览器访问：", vis.url())
display(vis.jupyter_cell())   # 也可内嵌在 notebook 里

# ── 坐标转换：机器人 Z-up → meshcat/Three.js Y-up ─────────────────────────────
def r2m(p):
    """(xr, yr, zr) → (xr, zr, -yr)"""
    return np.array([p[0], p[2], -p[1]], dtype=float)

# ── 辅助：在两点之间放置圆柱 ─────────────────────────────────────────────────
def _capsule(vis, name, p1, p2, radius, color):
    d = p2 - p1;  L = np.linalg.norm(d)
    if L < 1e-6: return
    y = np.array([0., 1., 0.])
    d_hat = d / L
    cross = np.cross(y, d_hat);  cn = np.linalg.norm(cross)
    if cn < 1e-6:
        R4 = np.eye(4) if d_hat[1] > 0 else np.diag([1., -1., -1., 1.])
    else:
        cross /= cn
        a = np.arccos(np.clip(np.dot(y, d_hat), -1, 1))
        K = np.array([[0,-cross[2],cross[1]],[cross[2],0,-cross[0]],[-cross[1],cross[0],0]])
        R3 = np.eye(3) + np.sin(a)*K + (1-np.cos(a))*K@K
        R4 = np.eye(4);  R4[:3,:3] = R3
    R4[:3, 3] = (p1 + p2) / 2
    vis[name].set_object(mc_g.Cylinder(L, radius),
                         mc_g.MeshPhongMaterial(color=color, reflectivity=0.5))
    vis[name].set_transform(R4)

def _sphere(vis, name, pos, r, color):
    vis[name].set_object(mc_g.Sphere(r), mc_g.MeshPhongMaterial(color=color))
    vis[name].set_transform(mc_tf.translation_matrix(pos))

# ── RRR 场景更新 ──────────────────────────────────────────────────────────────
_L1, _L2, _L3, _BH = 0.5, 0.4, 0.3, 0.15
LCOLORS = [0x1976d2, 0x388e3c, 0xf57c00]

def _update_mc(t1=30., t2=60., t3=-45.):
    t1r, t2r, t3r = np.radians([t1, t2, t3])
    p0 = np.array([0., 0., _BH])
    p1 = p0 + _L1 * np.array([np.cos(t1r),       np.sin(t1r),       0.])
    p2 = p1 + _L2 * np.array([np.cos(t1r+t2r),   np.sin(t1r+t2r),   0.])
    p3 = p2 + _L3 * np.array([np.cos(t1r+t2r+t3r), np.sin(t1r+t2r+t3r), 0.])
    pts = [r2m(p) for p in [p0, p1, p2, p3]]

    # 底座
    _capsule(vis, 'arm/base', r2m([0,0,0]), r2m([0,0,_BH]), 0.07, 0x2c3e50)
    # 连杆
    for i, (pa, pb, c) in enumerate(zip(pts, pts[1:], LCOLORS)):
        _capsule(vis, f'arm/link{i+1}', pa, pb, [0.048,0.040,0.032][i], c)
    # 关节球
    for i, (p, r) in enumerate(zip(pts, [0.068,0.058,0.048,0.040])):
        _sphere(vis, f'arm/joint{i}', p, r, 0xe8eaf6)
    # 末端执行器
    _sphere(vis, 'arm/ee', pts[-1], 0.045, 0xff3030)
    # 地板网格（只需画一次）
    vis['floor'].set_object(mc_g.Box([2, 0.002, 2]),
                            mc_g.MeshPhongMaterial(color=0x1a1f2e))

# ── 滑块 + observe 回调 ───────────────────────────────────────────────────────
_s1 = FloatSlider(value=30,  min=-180, max=180, step=5, description='θ₁ (°)', style={'description_width':'60px'})
_s2 = FloatSlider(value=60,  min=-180, max=180, step=5, description='θ₂ (°)', style={'description_width':'60px'})
_s3 = FloatSlider(value=-45, min=-180, max=180, step=5, description='θ₃ (°)', style={'description_width':'60px'})

def _on_change(_):
    _update_mc(_s1.value, _s2.value, _s3.value)

for _s in [_s1, _s2, _s3]:
    _s.observe(_on_change, names='value')

_update_mc()   # 初始渲染
display(HBox([_s1, _s2, _s3]))
print("✓ 拖动滑块 → 浏览器实时更新")

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/
浏览器访问： http://127.0.0.1:7000/static/


✓ 拖动滑块 → 浏览器实时更新


---
## Part 10：方案三 — Three.js 独立网页（rcfs.ch 同款技术栈）

生成一个独立 HTML 文件，用浏览器直接打开即可。  
技术栈与 rcfs.ch 完全相同：**Three.js + OrbitControls + 金属材质 + 阴影**。  
支持切换 RRR / PUMA 560 / RRRP，显示 DH 坐标轴。

In [14]:
import webbrowser, textwrap
from pathlib import Path

HTML = textwrap.dedent(r"""
<!DOCTYPE html><html lang="zh-CN"><head><meta charset="UTF-8">
<title>正运动学 — 台大机器人学</title>
<style>
:root{--bg:#0d1117;--panel:#161b22;--border:#21262d;--text:#c9d1d9;--muted:#8b949e;--acc:#58a6ff}
*{margin:0;padding:0;box-sizing:border-box}
body{height:100vh;display:flex;background:var(--bg);color:var(--text);
  font:13px/1.5 'Segoe UI',system-ui,sans-serif;overflow:hidden}
#panel{width:290px;min-width:290px;background:var(--panel);border-right:1px solid var(--border);
  display:flex;flex-direction:column;overflow:hidden}
.ph{padding:14px 16px;border-bottom:1px solid var(--border)}
.ph h1{font-size:15px;font-weight:600;color:var(--acc)}
.ph p{font-size:11px;color:var(--muted);margin-top:2px}
.tabs{display:flex;gap:6px;padding:10px 16px;border-bottom:1px solid var(--border)}
.tab{flex:1;padding:6px 4px;border:1px solid var(--border);background:transparent;color:var(--muted);
  border-radius:6px;cursor:pointer;font-size:11px;font-weight:500;transition:.15s}
.tab:hover{border-color:var(--acc);color:var(--text)}
.tab.on{background:var(--acc);border-color:var(--acc);color:#0d1117;font-weight:700}
#sliders{flex:1;overflow-y:auto;padding:4px 0}
.sr{padding:9px 16px}
.sh{display:flex;justify-content:space-between;margin-bottom:4px}
.jn{font-weight:500;font-size:13px}.jv{color:var(--acc);font-family:monospace;font-size:12px}
input[type=range]{width:100%;height:3px;accent-color:var(--acc);cursor:pointer}
.trow{display:flex;align-items:center;gap:8px;padding:8px 16px;border-top:1px solid var(--border)}
.sw{width:32px;height:18px;background:var(--border);border-radius:9px;cursor:pointer;position:relative;flex-shrink:0}
.sw.on{background:var(--acc)}
.sw::after{content:'';position:absolute;width:14px;height:14px;background:#fff;border-radius:50%;top:2px;left:2px;transition:left .15s}
.sw.on::after{left:16px}
#info{padding:12px 16px;background:rgba(0,0,0,.3);border-top:1px solid var(--border)}
.it{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:1px;margin-bottom:6px}
.ig{display:grid;grid-template-columns:auto 1fr;gap:3px 10px}
.ik{color:var(--muted);font-size:12px}.iv{color:var(--acc);font-family:monospace;font-size:12px;text-align:right}
#cv{flex:1;position:relative}canvas{display:block}
</style></head><body>
<div id="panel">
  <div class="ph"><h1>正运动学可视化</h1><p>台大林沛群教授《机器人学》第三章</p></div>
  <div class="tabs">
    <button class="tab on" id="tb-rrr"  onclick="setR('rrr')">RRR</button>
    <button class="tab"    id="tb-puma" onclick="setR('puma')">PUMA 560</button>
    <button class="tab"    id="tb-rrrp" onclick="setR('rrrp')">RRRP</button>
  </div>
  <div id="sliders"></div>
  <div class="trow"><div class="sw" id="sw-ax" onclick="togAx()"></div>
    <span style="font-size:12px;color:var(--muted)">显示坐标轴</span></div>
  <div id="info">
    <div class="it">末端执行器</div>
    <div class="ig">
      <span class="ik">x</span><span class="iv" id="vx">—</span>
      <span class="ik">y</span><span class="iv" id="vy">—</span>
      <span class="ik">z</span><span class="iv" id="vz">—</span>
      <span class="ik" id="lp">φ</span><span class="iv" id="vp">—</span>
    </div>
  </div>
</div>
<div id="cv"></div>

<script type="importmap">
{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
"three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}
</script>
<script type="module">
import * as T from 'three';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

// ── Renderer ──
const wrap=document.getElementById('cv');
const ren=new T.WebGLRenderer({antialias:true});
ren.setPixelRatio(devicePixelRatio);
ren.shadowMap.enabled=true; ren.shadowMap.type=T.PCFSoftShadowMap;
ren.toneMapping=T.ACESFilmicToneMapping; ren.toneMappingExposure=1.15;
wrap.appendChild(ren.domElement);
const scene=new T.Scene(); scene.background=new T.Color(0x0d1117);
scene.fog=new T.FogExp2(0x0d1117,.12);
const cam=new T.PerspectiveCamera(42,1,.01,50);
cam.position.set(2.2,1.8,2.2);
const ctrl=new OrbitControls(cam,ren.domElement);
ctrl.enableDamping=true; ctrl.dampingFactor=.06;
ctrl.target.set(0,.4,0); ctrl.minDistance=.4; ctrl.maxDistance=10;
function resize(){const w=wrap.clientWidth,h=wrap.clientHeight;
  ren.setSize(w,h); cam.aspect=w/h; cam.updateProjectionMatrix();}
window.addEventListener('resize',resize); resize();

// ── Lights ──
scene.add(new T.AmbientLight(0x8090b0,.9));
const sun=new T.DirectionalLight(0xffffff,3.5);
sun.position.set(3,4,3); sun.castShadow=true;
sun.shadow.mapSize.set(2048,2048);
Object.assign(sun.shadow.camera,{left:-2,right:2,top:2,bottom:-2,near:.1,far:20});
scene.add(sun);
const fill=new T.DirectionalLight(0x8090ff,.6); fill.position.set(-2,2,-1); scene.add(fill);

// ── Floor ──
const fm=new T.MeshStandardMaterial({color:0x1a1f2e,roughness:.9});
const fl=new T.Mesh(new T.PlaneGeometry(8,8),fm);
fl.rotation.x=-Math.PI/2; fl.receiveShadow=true; scene.add(fl);
scene.add(new T.GridHelper(4,20,0x2a2d40,0x1e2130));

// ── Materials ──
const MAT={
  base:new T.MeshStandardMaterial({color:0x2c3e50,roughness:.4,metalness:.75}),
  joint:new T.MeshStandardMaterial({color:0xe8eaf6,roughness:.15,metalness:.9}),
  ee:new T.MeshStandardMaterial({color:0xff3030,roughness:.1,metalness:.9,emissive:0x440000}),
  lk:[0x1976d2,0x388e3c,0xf57c00,0x7b1fa2,0x00838f,0xc62828].map(
    c=>new T.MeshStandardMaterial({color:c,roughness:.3,metalness:.6}))
};

// ── FK Math (Modified DH, row-major set) ──
function mdh(a,aDeg,d,tDeg){
  const al=aDeg*Math.PI/180, th=tDeg*Math.PI/180;
  const [ca,sa,ct,st]=[Math.cos(al),Math.sin(al),Math.cos(th),Math.sin(th)];
  const M=new T.Matrix4();
  M.set(ct,-st,0,a, st*ca,ct*ca,-sa,-d*sa, st*sa,ct*sa,ca,d*ca, 0,0,0,1);
  return M;}
function chainFK(rows){let T0=new T.Matrix4(); const fs=[T0.clone()];
  for(const r of rows){T0=T0.clone().multiply(mdh(...r)); fs.push(T0.clone());}
  return fs;}
function pos3(M){const e=M.elements; return new T.Vector3(e[12],e[13],e[14]);}
// Z-up → Y-up:  (xr,yr,zr) → (xr, zr, -yr)
function r2t(v,bh=0){return new T.Vector3(v.x,v.z+bh,-v.y);}

// ── Robot Configs ──
const ROBOTS={
  rrr:{joints:[{l:'θ₁',mn:-180,mx:180,v:30},{l:'θ₂',mn:-180,mx:180,v:60},{l:'θ₃',mn:-180,mx:180,v:-45}],
    dh:a=>[[0,0,0,a[0]],[.5,0,0,a[1]],[.4,0,0,a[2]],[.3,0,0,0]],
    radii:[.055,.045,.035,0], bh:.12, phi:true},
  puma:{joints:[{l:'θ₁',mn:-180,mx:180,v:0},{l:'θ₂',mn:-135,mx:135,v:90},
    {l:'θ₃',mn:-135,mx:135,v:-90},{l:'θ₄',mn:-180,mx:180,v:0},
    {l:'θ₅',mn:-135,mx:135,v:90},{l:'θ₆',mn:-180,mx:180,v:0}],
    dh:a=>[[0,0,0,a[0]],[0,-90,0,a[1]],[.4318,0,0,a[2]],
           [-.0203,-90,.4331,a[3]],[0,90,0,a[4]],[0,-90,0,a[5]]],
    radii:[.055,.048,.042,.035,.030,.024], bh:.15, phi:false},
  rrrp:{joints:[{l:'θ₁',mn:-180,mx:180,v:40},{l:'θ₂',mn:-180,mx:180,v:60},
    {l:'θ₃',mn:-180,mx:180,v:-30},{l:'d₄',mn:0,mx:.6,v:.2,step:.01,unit:'m'}],
    dh:a=>[[0,0,0,a[0]],[.45,0,0,a[1]],[.35,0,0,a[2]],[.25,0,a[3],0]],
    radii:[.055,.045,.038,.028], bh:.12, phi:true}
};

// ── Scene Graph ──
const root=new T.Group(); scene.add(root);
let objs={joints:[],links:[],axH:[],ee:null};
const Y1=new T.Vector3(0,1,0);

function mkCap(r,l){return new T.CapsuleGeometry(r,Math.max(.001,l),10,20);}
function rebuild(cfg){
  while(root.children.length)root.remove(root.children[0]);
  objs={joints:[],links:[],axH:[],ee:null};
  const n=cfg.joints.length, bh=cfg.bh, rads=cfg.radii;
  // Base
  const bc=new T.Mesh(new T.CylinderGeometry(.10,.12,bh,32),MAT.base);
  bc.position.y=bh/2; bc.castShadow=bc.receiveShadow=true; root.add(bc);
  const br=new T.Mesh(new T.CylinderGeometry(.09,.09,.015,32),MAT.joint);
  br.position.y=bh+.007; root.add(br);
  for(let i=0;i<n;i++){
    const r=rads[i]||.025;
    const jm=new T.Mesh(new T.SphereGeometry(r*1.5,32,32),MAT.joint.clone());
    jm.castShadow=true; root.add(jm); objs.joints.push(jm);
    const lm=new T.Mesh(mkCap(r,.1),MAT.lk[i%6].clone());
    lm.castShadow=true; root.add(lm); objs.links.push(lm);
    const ah=new T.AxesHelper(.16); ah.visible=false; root.add(ah); objs.axH.push(ah);
  }
  const em=new T.Mesh(new T.SphereGeometry(.042,32,32),MAT.ee.clone());
  em.castShadow=true; root.add(em); objs.ee=em;
}

function setLink(mesh,p1,p2,r){
  const dir=new T.Vector3().subVectors(p2,p1), L=dir.length();
  if(L<.001)return;
  mesh.position.copy(p1.clone().add(p2).multiplyScalar(.5));
  mesh.quaternion.setFromUnitVectors(Y1,dir.normalize());
  mesh.geometry.dispose(); mesh.geometry=mkCap(r,L-r*1.5);
}

let curRobot='rrr', showAx=false;

function updatePose(angles){
  const cfg=ROBOTS[curRobot];
  const frames=chainFK(cfg.dh(angles));
  const bh=cfg.bh, bv=new T.Vector3(0,bh,0);
  for(let i=0;i<objs.joints.length;i++){
    const p=r2t(pos3(frames[i]),bh);
    objs.joints[i].position.copy(p);
    objs.axH[i].position.copy(p); objs.axH[i].visible=showAx;
    const pn=r2t(pos3(frames[i+1]),bh);
    setLink(objs.links[i],p,pn,cfg.radii[i]||.025);
  }
  const ee=r2t(pos3(frames[frames.length-1]),bh);
  objs.ee.position.copy(ee);
  const er=pos3(frames[frames.length-1]);
  document.getElementById('vx').textContent=er.x.toFixed(4);
  document.getElementById('vy').textContent=er.y.toFixed(4);
  document.getElementById('vz').textContent=er.z.toFixed(4);
  if(cfg.phi&&angles.length>=3){
    document.getElementById('lp').textContent='φ=θ₁+θ₂+θ₃';
    document.getElementById('vp').textContent=(angles[0]+angles[1]+angles[2]).toFixed(1)+'°';
  }else{document.getElementById('vp').textContent='—';}
}

function buildSliders(cfg){
  const c=document.getElementById('sliders'); c.innerHTML='';
  cfg.joints.forEach((j,i)=>{
    const step=j.step||1, unit=j.unit||'°';
    const d=document.createElement('div'); d.className='sr';
    d.innerHTML=`<div class="sh"><span class="jn">${j.l}</span>
      <span class="jv" id="jv${i}">${j.v.toFixed(j.unit?2:0)}${unit}</span></div>
      <input type="range" id="js${i}" min="${j.mn}" max="${j.mx}" step="${step}" value="${j.v}">`;
    d.querySelector('input').addEventListener('input',()=>{
      const a=ROBOTS[curRobot].joints.map((_,k)=>{
        const el=document.getElementById('js'+k); return el?parseFloat(el.value):0;});
      ROBOTS[curRobot].joints.forEach((_,k)=>{
        const u=ROBOTS[curRobot].joints[k].unit||'°';
        const el=document.getElementById('jv'+k); if(el)el.textContent=a[k].toFixed(u==='m'?2:0)+u;});
      updatePose(a);});
    c.appendChild(d);});
}

window.setR=function(name){
  curRobot=name;
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('on'));
  document.getElementById('tb-'+name).classList.add('on');
  const cfg=ROBOTS[name]; buildSliders(cfg); rebuild(cfg);
  updatePose(cfg.joints.map(j=>j.v));};

window.togAx=function(){
  showAx=!showAx;
  document.getElementById('sw-ax').classList.toggle('on',showAx);
  objs.axH.forEach(a=>{a.visible=showAx;});};

setR('rrr');
(function anim(){requestAnimationFrame(anim); ctrl.update(); ren.render(scene,cam);})();
</script></body></html>
""").strip()

def _resolve_notebook_dir():
    cwd = Path.cwd()
    for base in (cwd, cwd / 'Robotics_NTU'):
        if (base / 'Chapter03_visualization.ipynb').exists():
            return base
    return cwd


# 写入文件
NOTEBOOK_DIR = _resolve_notebook_dir()
out_html = NOTEBOOK_DIR / 'robot_fk_viewer.html'
out_html.write_text(HTML, encoding='utf-8')
print(f"✓ 已写入：{out_html}")
print("  → 在浏览器中打开该文件即可使用（需要联网加载 Three.js CDN）")

# 自动在浏览器打开
webbrowser.open(out_html.as_uri())
print("  → 正在自动打开浏览器...")

✓ 已写入：d:\EI-Beginner\Robotics_NTU\robot_fk_viewer.html
  → 在浏览器中打开该文件即可使用（需要联网加载 Three.js CDN）
  → 正在自动打开浏览器...
